In [ ]:
%pip install pymorphy3

import pandas as pd
from collections import Counter, defaultdict
from pymorphy3 import MorphAnalyzer

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.4 MB 6.3 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.4 MB 10.2 MB/s eta 0:00:01
   -------------------------- ------------- 5.5/8.4 MB 10.5 MB/s eta 0:00:01
   -------------------------------------- - 8.1/8.4 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 10.4 MB/s  0:00:00

   -------------------------- ------------- 2/3 [pymorphy3]
   -------------------------- ------------- 2/3 [pymorphy3]
   -------------------------- ------------- 2/3 [pymorphy3]
   ---------------------------------------- 3/3 [pymorphy3]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
df = pd.read_csv(r"C:\Users\тема\Desktop\Реплики.csv", sep=';')
df = df.iloc[:, :2]
df.columns = ["Реплика", "Эмоция"]
df = df.dropna()

In [14]:
#Глагольные признаки
features_map = {
    "Род": defaultdict(Counter),
    "Время": defaultdict(Counter),
    "Лицо": defaultdict(Counter),
    "Число": defaultdict(Counter),
    "Наклонение": defaultdict(Counter),
}

for _, row in df.iterrows():
    text = str(row["Реплика"])
    emotion = str(row["Эмоция"]).strip()
    for w in text.split():
        clean = w.strip('.,!?;:()""«»—…')
        if not clean:
            continue
        p = morph.parse(clean)[0]
        tag = p.tag
        if tag.POS in ('VERB', 'INFN'):
            features_map["Род"][tag.gender or '-'][emotion] += 1
            features_map["Время"][tag.tense or '-'][emotion] += 1
            features_map["Лицо"][tag.person or '-'][emotion] += 1
            features_map["Число"][tag.number or '-'][emotion] += 1
            features_map["Наклонение"][tag.mood or '-'][emotion] += 1

for feature_name, data in features_map.items():
    total_all = sum(sum(c.values()) for c in data.values())
    print(f"\n{'-'*60}")
    print(f"  {feature_name}")
    print(f"{'-'*60}")
    for value, counts in sorted(data.items(), key=lambda x: -sum(x[1].values())):
        total = sum(counts.values())
        pct = total / total_all * 100 if total_all > 0 else 0
        top_str = ", ".join(f"{emo}: {cnt}" for emo, cnt in counts.most_common())
        print(f"\n  {value}: {total} ({pct:.1f}%)")
        print(f"    {top_str}")

#Предложения с отрицанием
ne_stats = Counter()
ne_total = 0
for _, row in df.iterrows():
    tokens = str(row["Реплика"]).lower().split()
    if "не" in tokens:
        emotion = str(row["Эмоция"]).strip()
        ne_stats[emotion] += 1
        ne_total += 1

print(f"\n{'-'*60}")
print(f"  Предложения с 'не'  (всего: {ne_total})")
print(f"{'-'*60}")
for emo, cnt in ne_stats.most_common():
    pct = cnt / ne_total * 100 if ne_total > 0 else 0
    print(f"  {emo}: {cnt} ({pct:.1f}%)")

#Знаки препинания в конце 
for punct, label in [("?", "вопросительные"), ("!", "восклицательные"), (".", "утвердительные")]:
    punct_stats = Counter()
    punct_total = 0
    for _, row in df.iterrows():
        text = str(row["Реплика"]).strip()
        emotion = str(row["Эмоция"]).strip()
        if text.endswith(punct):
            punct_stats[emotion] += 1
            punct_total += 1
    print(f"\n{'-'*60}")
    print(f"  Предложения, оканчивающиеся на '{punct}' ({label})  (всего: {punct_total})")
    print(f"{'-'*60}")
    for emo, cnt in punct_stats.most_common():
        pct = cnt / punct_total * 100 if punct_total > 0 else 0
        print(f"  {emo}: {cnt} ({pct:.1f}%)")


------------------------------------------------------------
  Род
------------------------------------------------------------

  femn: 2796 (50.2%)
    Joy: 722, Sadness: 607, Surprise: 591, Anger: 363, Fear: 271, Neutral: 220, Неграмматично: 22

  -: 2771 (49.8%)
    Anger: 567, Sadness: 555, Joy: 549, Surprise: 517, Fear: 306, Neutral: 234, Неграмматично: 43

------------------------------------------------------------
  Время
------------------------------------------------------------

  past: 2796 (50.2%)
    Joy: 722, Sadness: 607, Surprise: 591, Anger: 363, Fear: 271, Neutral: 220, Неграмматично: 22

  -: 2760 (49.6%)
    Anger: 566, Sadness: 554, Joy: 547, Surprise: 515, Fear: 303, Neutral: 233, Неграмматично: 42

  futr: 11 (0.2%)
    Fear: 3, Surprise: 2, Joy: 2, Neutral: 1, Неграмматично: 1, Anger: 1, Sadness: 1

------------------------------------------------------------
  Лицо
------------------------------------------------------------

  -: 5556 (99.8%)
    Joy: 1269